<a href="https://colab.research.google.com/github/ElofssonLab/kb8029-book/blob/main/notebooks/day04-lab.ipynb" style="display:inline-block;padding:10px 18px;background-color:#F9AB00;color:#000000;font-weight:bold;text-decoration:none;border-radius:6px;font-family:sans-serif;font-size:14px;">&#9654;&nbsp; Open in Google Colab</a>

# Day 4 lab: a manual sequence alignment, by hand {.unnumbered}

Three steps:

1. **By hand** (on paper, or in your head, not in this notebook): fill in
   a Needleman-Wunsch dynamic-programming matrix and trace back the
   optimal alignment, using a real substitution matrix. This notebook
   only *checks* your work, one small piece at a time -- it never
   computes the matrix for you.
2. **In code**: see how the score and the optimal alignment change as
   you vary the gap penalty and the substitution matrix -- first with
   the same small DP recurrence from Step 1, then with a real local
   BLAST search, to see how E-values (a database-search concept, not a
   two-sequence one) respond to the same kinds of changes.
3. **In code**: a small, real multiple sequence alignment -- built and
   visualized directly, not just described.

Once all three steps are done, answer the Day 4 lab quiz on Canvas using
your own results.


## Setup: the exercise for Step 1

Align two real, short protein fragments by hand:

- **seqA = "HFGKE"** -- human beta-globin (HBB, UniProt `P68871`), residues 118-122
- **seqB = "HGQE"** -- human myoglobin (MYG, UniProt `P02144`), residues 25-28

Use a flat, linear gap penalty of **-4** per gap position, and the real
**BLOSUM62** substitution matrix (Henikoff & Henikoff, 1992) below,
restricted to just the residues that appear in these two fragments:

|     | E  | F  | G  | H  | K  | Q  |
|-----|---:|---:|---:|---:|---:|---:|
| **E** | 5 | -3 | -2 | 0 | 1 | 2 |
| **F** | -3 | 6 | -3 | -1 | -3 | -3 |
| **G** | -2 | -3 | 6 | -2 | -2 | -2 |
| **H** | 0 | -1 | -2 | 8 | -1 | 0 |
| **K** | 1 | -3 | -2 | -1 | 5 | 1 |
| **Q** | 2 | -3 | -2 | 0 | 1 | 5 |

The recurrence (same as the book page and the teaching notebook):

$$
S[i,j] = \max \begin{cases} S[i-1,j-1] + s(x_i, y_j) & \text{diagonal: match/mismatch} \\ S[i-1,j] + \text{gap} & \text{up: gap in seqB} \\ S[i,j-1] + \text{gap} & \text{left: gap in seqA} \end{cases}
$$

Run the setup cell below to get these same values programmatically (and
the mechanical border row/column already filled in) -- reading them
here first just means you don't have to execute anything to know what
the exercise even is.


In [ ]:
import numpy as np

# Two real protein fragments -- not the same ones used anywhere else on
# the Day 4 page or its teaching notebook:
#   seqA = "HFGKE", human beta-globin (HBB, UniProt P68871), residues 118-122
#   seqB = "HGQE",  human myoglobin  (MYG, UniProt P02144), residues 25-28
seqA, seqB = "HFGKE", "HGQE"
gap = -4.0

# The real BLOSUM62 substitution matrix (Henikoff & Henikoff, 1992),
# restricted to just the residues that appear in seqA/seqB above --
# hardcoded here (rather than loaded via Biopython's
# Bio.Align.substitution_matrices) so this notebook has no dependency
# beyond NumPy/Matplotlib and runs in a stock Colab runtime with no
# `!pip install` step. Values confirmed against
# Bio.Align.substitution_matrices.load("BLOSUM62") while building this
# notebook.
BLOSUM62_SUB = {
    ('E', 'E'): 5, ('E', 'F'): -3, ('E', 'G'): -2, ('E', 'H'): 0, ('E', 'K'): 1, ('E', 'Q'): 2,
    ('F', 'E'): -3, ('F', 'F'): 6, ('F', 'G'): -3, ('F', 'H'): -1, ('F', 'K'): -3, ('F', 'Q'): -3,
    ('G', 'E'): -2, ('G', 'F'): -3, ('G', 'G'): 6, ('G', 'H'): -2, ('G', 'K'): -2, ('G', 'Q'): -2,
    ('H', 'E'): 0, ('H', 'F'): -1, ('H', 'G'): -2, ('H', 'H'): 8, ('H', 'K'): -1, ('H', 'Q'): 0,
    ('K', 'E'): 1, ('K', 'F'): -3, ('K', 'G'): -2, ('K', 'H'): -1, ('K', 'K'): 5, ('K', 'Q'): 1,
    ('Q', 'E'): 2, ('Q', 'F'): -3, ('Q', 'G'): -2, ('Q', 'H'): 0, ('Q', 'K'): 1, ('Q', 'Q'): 5,
}
blosum62 = BLOSUM62_SUB
residues = sorted(set(seqA) | set(seqB))

def score(a, b, subst=blosum62, gap_penalty=gap):
    if a == '-' or b == '-':
        return gap_penalty
    return subst[a, b]

print(f"seqA = {seqA!r}  ({len(seqA)} residues, human beta-globin fragment)")
print(f"seqB = {seqB!r}  ({len(seqB)} residues, human myoglobin fragment)")
print(f"gap penalty = {gap} (flat, linear -- same recurrence as the book page and teaching notebook)")
print()
print("BLOSUM62, restricted to just the residues appearing above (real values):")
print("     " + "  ".join(f"{r:>2}" for r in residues))
for r1 in residues:
    print(f"{r1:>2}  " + "  ".join(f"{blosum62[r1, r2]:>2}" for r2 in residues))
print()

m, n = len(seqA) + 1, len(seqB) + 1
S = np.full((m, n), np.nan)
for i in range(m):
    S[i, 0] = i * gap
for j in range(n):
    S[0, j] = j * gap

print("Border filled in (mechanical -- i*gap / j*gap), interior left for you to work out by hand:")
print("Rows = '-' + seqA, columns = '-' + seqB:")
print("        " + "     ".join(["-"] + list(seqB)))
for i, row_label in enumerate(["-"] + list(seqA)):
    print(row_label, " ", S[i])


## Step 1: fill in the matrix, one row at a time

Work out each row of the interior matrix by hand, then fill in that
row's cell below and run it to check just that row -- rather than
committing to all 20 values before getting any feedback at all. Row $i$
corresponds to `seqA[i-1]`, e.g. row 1 is `seqA[0] = 'H'`.


In [ ]:
# Row 1 (seqA[0] = 'H'). Fill in S[1,1] through S[1,4].
row_1_guesses = {
    (1, 1): None, (1, 2): None, (1, 3): None, (1, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking row 1:")
_all_correct = True
for (i, j), guess in row_1_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        _all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        _all_correct = False
print("Row 1 all correct -- move on to the next row." if _all_correct
      else "Fix row 1 and re-run this cell before moving on.")


In [ ]:
# Row 2 (seqA[1] = 'F'). Fill in S[2,1] through S[2,4].
row_2_guesses = {
    (2, 1): None, (2, 2): None, (2, 3): None, (2, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking row 2:")
_all_correct = True
for (i, j), guess in row_2_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        _all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        _all_correct = False
print("Row 2 all correct -- move on to the next row." if _all_correct
      else "Fix row 2 and re-run this cell before moving on.")


In [ ]:
# Row 3 (seqA[2] = 'G'). Fill in S[3,1] through S[3,4].
row_3_guesses = {
    (3, 1): None, (3, 2): None, (3, 3): None, (3, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking row 3:")
_all_correct = True
for (i, j), guess in row_3_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        _all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        _all_correct = False
print("Row 3 all correct -- move on to the next row." if _all_correct
      else "Fix row 3 and re-run this cell before moving on.")


In [ ]:
# Row 4 (seqA[3] = 'K'). Fill in S[4,1] through S[4,4].
row_4_guesses = {
    (4, 1): None, (4, 2): None, (4, 3): None, (4, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking row 4:")
_all_correct = True
for (i, j), guess in row_4_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        _all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        _all_correct = False
print("Row 4 all correct -- move on to the next row." if _all_correct
      else "Fix row 4 and re-run this cell before moving on.")


In [ ]:
# Row 5 (seqA[4] = 'E'). Fill in S[5,1] through S[5,4].
row_5_guesses = {
    (5, 1): None, (5, 2): None, (5, 3): None, (5, 4): None,
}

def _real_matrix():
    real = S.copy()
    for i in range(1, m):
        for j in range(1, n):
            diag = real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = real[i-1, j] + gap
            left = real[i, j-1] + gap
            real[i, j] = max(diag, up, left)
    return real

_real = _real_matrix()

print("Checking row 5:")
_all_correct = True
for (i, j), guess in row_5_guesses.items():
    real_val = _real[i, j]
    if guess is None:
        print(f"  S[{i},{j}]: not filled in yet")
        _all_correct = False
    elif float(guess) == real_val:
        print(f"  S[{i},{j}]: correct! ({real_val:.0f})")
    else:
        print(f"  S[{i},{j}]: your guess {guess} does not match the real value {real_val:.0f}")
        _all_correct = False
print("Row 5 all correct -- move on to the next row." if _all_correct
      else "Fix row 5 and re-run this cell before moving on.")


### Now trace back

Once every row above checks out, trace back from the bottom-right
corner to the top-left, following the winning move at each cell, and
write out the two final aligned strings.


In [ ]:
# Write out the two final aligned strings (use '-' for a gap). Format
# only -- this example is NOT the real answer for this exercise, just
# shows the shape expected: e.g. your_alignment = ("PQR", "P-R") would
# mean no gap in seqA and one gap in seqB (at the middle position) for a
# 3-letter seqA aligned to a 2-letter seqB. Work out the real one yourself.
your_alignment = (None, None)

def _real_traceback():
    trace = np.zeros((m, n, 2))
    for i in range(1, m):
        trace[i, 0, :] = (-1, 0)
    for j in range(1, n):
        trace[0, j, :] = (0, -1)
    for i in range(1, m):
        for j in range(1, n):
            diag = _real[i-1, j-1] + score(seqA[i-1], seqB[j-1])
            up = _real[i-1, j] + gap
            left = _real[i, j-1] + gap
            if diag >= max(up, left):
                trace[i, j, :] = (-1, -1)
            elif up >= left:
                trace[i, j, :] = (-1, 0)
            else:
                trace[i, j, :] = (0, -1)
    outA, outB = "", ""
    i, j = len(seqA), len(seqB)
    while i > 0 or j > 0:
        di, dj = trace[i, j]
        i, j = i + int(di), j + int(dj)
        outA = ("-" if di == 0 else seqA[i]) + outA
        outB = ("-" if dj == 0 else seqB[j]) + outB
    return outA, outB

real_outA, real_outB = _real_traceback()

guessA, guessB = your_alignment
if guessA is None or guessB is None:
    print("Not filled in yet.")
elif guessA == real_outA and guessB == real_outB:
    print("Correct! Your traceback matches the real optimal alignment.")
    print(guessA)
    print(guessB)
else:
    print("Not quite -- the real alignment is:")
    print(real_outA)
    print(real_outB)
    print("(yours was:)")
    print(guessA)
    print(guessB)


Once the row checks and the traceback check above all say "correct,"
Step 1 is done -- you have the full matrix, the final score, and the
final alignment.


## Step 2: how do the score and alignment change with different parameters?

This step is exploratory, not another by-hand check -- run the cells
and look at real results. Same `seqA`/`seqB` as Step 1, but now scored
in code across a few different gap penalties and substitution matrices,
to see directly how much the choice of parameters actually matters.


In [ ]:
def global_align(seqA, seqB, gap, subst):
    """Same Needleman-Wunsch recurrence as Step 1 and the teaching notebook,
    generalized to take any gap penalty / substitution matrix."""
    m, n = len(seqA) + 1, len(seqB) + 1
    S = np.zeros((m, n))
    trace = np.zeros((m, n, 2))
    for i in range(1, m):
        S[i, 0] = i * gap
        trace[i, 0, :] = (-1, 0)
    for j in range(1, n):
        S[0, j] = j * gap
        trace[0, j, :] = (0, -1)
    for i in range(1, m):
        for j in range(1, n):
            diag = S[i-1, j-1] + subst(seqA[i-1], seqB[j-1])
            up = S[i-1, j] + gap
            left = S[i, j-1] + gap
            S[i, j] = max(diag, up, left)
            if diag >= max(up, left):
                trace[i, j, :] = (-1, -1)
            elif up >= left:
                trace[i, j, :] = (-1, 0)
            else:
                trace[i, j, :] = (0, -1)
    return S, trace

def traceback(seqA, seqB, S, trace):
    i, j = len(seqA), len(seqB)
    outA, outB = "", ""
    while i > 0 or j > 0:
        di, dj = trace[i, j]
        i, j = i + int(di), j + int(dj)
        outA = ("-" if di == 0 else seqA[i]) + outA
        outB = ("-" if dj == 0 else seqB[j]) + outB
    return outA, outB

# The real PAM250 substitution matrix (Dayhoff, Schwartz & Orcutt, 1978),
# restricted to the same 6 residues -- values confirmed against
# Bio.Align.substitution_matrices.load("PAM250"), same approach as
# BLOSUM62 above (hardcoded, no Biopython dependency).
PAM250_SUB = {
    ('E', 'E'): 4, ('E', 'F'): -5, ('E', 'G'): 0, ('E', 'H'): 1, ('E', 'K'): 0, ('E', 'Q'): 2,
    ('F', 'E'): -5, ('F', 'F'): 9, ('F', 'G'): -5, ('F', 'H'): -2, ('F', 'K'): -5, ('F', 'Q'): -5,
    ('G', 'E'): 0, ('G', 'F'): -5, ('G', 'G'): 5, ('G', 'H'): -2, ('G', 'K'): -2, ('G', 'Q'): -1,
    ('H', 'E'): 1, ('H', 'F'): -2, ('H', 'G'): -2, ('H', 'H'): 6, ('H', 'K'): 0, ('H', 'Q'): 3,
    ('K', 'E'): 0, ('K', 'F'): -5, ('K', 'G'): -2, ('K', 'H'): 0, ('K', 'K'): 5, ('K', 'Q'): 1,
    ('Q', 'E'): 2, ('Q', 'F'): -5, ('Q', 'G'): -1, ('Q', 'H'): 3, ('Q', 'K'): 1, ('Q', 'Q'): 4,
}

# A wider BLOSUM62 sub-table (17 residues) -- Step 1's small 6-residue
# BLOSUM62_SUB above only covers seqA/seqB's own letters; Step 3's MSA
# below uses more residues, so this covers both. Same approach: hardcoded,
# confirmed against Bio.Align.substitution_matrices.load("BLOSUM62").
BLOSUM62_WIDE = {
    ('A', 'A'): 4, ('A', 'D'): -2, ('A', 'E'): -1, ('A', 'F'): -2, ('A', 'G'): 0, ('A', 'H'): -2, ('A', 'I'): -1, ('A', 'K'): -1, ('A', 'L'): -1, ('A', 'N'): -2, ('A', 'P'): -1, ('A', 'Q'): -1, ('A', 'S'): 1, ('A', 'T'): 0, ('A', 'V'): 0, ('A', 'W'): -3, ('A', 'Y'): -2,
    ('D', 'A'): -2, ('D', 'D'): 6, ('D', 'E'): 2, ('D', 'F'): -3, ('D', 'G'): -1, ('D', 'H'): -1, ('D', 'I'): -3, ('D', 'K'): -1, ('D', 'L'): -4, ('D', 'N'): 1, ('D', 'P'): -1, ('D', 'Q'): 0, ('D', 'S'): 0, ('D', 'T'): -1, ('D', 'V'): -3, ('D', 'W'): -4, ('D', 'Y'): -3,
    ('E', 'A'): -1, ('E', 'D'): 2, ('E', 'E'): 5, ('E', 'F'): -3, ('E', 'G'): -2, ('E', 'H'): 0, ('E', 'I'): -3, ('E', 'K'): 1, ('E', 'L'): -3, ('E', 'N'): 0, ('E', 'P'): -1, ('E', 'Q'): 2, ('E', 'S'): 0, ('E', 'T'): -1, ('E', 'V'): -2, ('E', 'W'): -3, ('E', 'Y'): -2,
    ('F', 'A'): -2, ('F', 'D'): -3, ('F', 'E'): -3, ('F', 'F'): 6, ('F', 'G'): -3, ('F', 'H'): -1, ('F', 'I'): 0, ('F', 'K'): -3, ('F', 'L'): 0, ('F', 'N'): -3, ('F', 'P'): -4, ('F', 'Q'): -3, ('F', 'S'): -2, ('F', 'T'): -2, ('F', 'V'): -1, ('F', 'W'): 1, ('F', 'Y'): 3,
    ('G', 'A'): 0, ('G', 'D'): -1, ('G', 'E'): -2, ('G', 'F'): -3, ('G', 'G'): 6, ('G', 'H'): -2, ('G', 'I'): -4, ('G', 'K'): -2, ('G', 'L'): -4, ('G', 'N'): 0, ('G', 'P'): -2, ('G', 'Q'): -2, ('G', 'S'): 0, ('G', 'T'): -2, ('G', 'V'): -3, ('G', 'W'): -2, ('G', 'Y'): -3,
    ('H', 'A'): -2, ('H', 'D'): -1, ('H', 'E'): 0, ('H', 'F'): -1, ('H', 'G'): -2, ('H', 'H'): 8, ('H', 'I'): -3, ('H', 'K'): -1, ('H', 'L'): -3, ('H', 'N'): 1, ('H', 'P'): -2, ('H', 'Q'): 0, ('H', 'S'): -1, ('H', 'T'): -2, ('H', 'V'): -3, ('H', 'W'): -2, ('H', 'Y'): 2,
    ('I', 'A'): -1, ('I', 'D'): -3, ('I', 'E'): -3, ('I', 'F'): 0, ('I', 'G'): -4, ('I', 'H'): -3, ('I', 'I'): 4, ('I', 'K'): -3, ('I', 'L'): 2, ('I', 'N'): -3, ('I', 'P'): -3, ('I', 'Q'): -3, ('I', 'S'): -2, ('I', 'T'): -1, ('I', 'V'): 3, ('I', 'W'): -3, ('I', 'Y'): -1,
    ('K', 'A'): -1, ('K', 'D'): -1, ('K', 'E'): 1, ('K', 'F'): -3, ('K', 'G'): -2, ('K', 'H'): -1, ('K', 'I'): -3, ('K', 'K'): 5, ('K', 'L'): -2, ('K', 'N'): 0, ('K', 'P'): -1, ('K', 'Q'): 1, ('K', 'S'): 0, ('K', 'T'): -1, ('K', 'V'): -2, ('K', 'W'): -3, ('K', 'Y'): -2,
    ('L', 'A'): -1, ('L', 'D'): -4, ('L', 'E'): -3, ('L', 'F'): 0, ('L', 'G'): -4, ('L', 'H'): -3, ('L', 'I'): 2, ('L', 'K'): -2, ('L', 'L'): 4, ('L', 'N'): -3, ('L', 'P'): -3, ('L', 'Q'): -2, ('L', 'S'): -2, ('L', 'T'): -1, ('L', 'V'): 1, ('L', 'W'): -2, ('L', 'Y'): -1,
    ('N', 'A'): -2, ('N', 'D'): 1, ('N', 'E'): 0, ('N', 'F'): -3, ('N', 'G'): 0, ('N', 'H'): 1, ('N', 'I'): -3, ('N', 'K'): 0, ('N', 'L'): -3, ('N', 'N'): 6, ('N', 'P'): -2, ('N', 'Q'): 0, ('N', 'S'): 1, ('N', 'T'): 0, ('N', 'V'): -3, ('N', 'W'): -4, ('N', 'Y'): -2,
    ('P', 'A'): -1, ('P', 'D'): -1, ('P', 'E'): -1, ('P', 'F'): -4, ('P', 'G'): -2, ('P', 'H'): -2, ('P', 'I'): -3, ('P', 'K'): -1, ('P', 'L'): -3, ('P', 'N'): -2, ('P', 'P'): 7, ('P', 'Q'): -1, ('P', 'S'): -1, ('P', 'T'): -1, ('P', 'V'): -2, ('P', 'W'): -4, ('P', 'Y'): -3,
    ('Q', 'A'): -1, ('Q', 'D'): 0, ('Q', 'E'): 2, ('Q', 'F'): -3, ('Q', 'G'): -2, ('Q', 'H'): 0, ('Q', 'I'): -3, ('Q', 'K'): 1, ('Q', 'L'): -2, ('Q', 'N'): 0, ('Q', 'P'): -1, ('Q', 'Q'): 5, ('Q', 'S'): 0, ('Q', 'T'): -1, ('Q', 'V'): -2, ('Q', 'W'): -2, ('Q', 'Y'): -1,
    ('S', 'A'): 1, ('S', 'D'): 0, ('S', 'E'): 0, ('S', 'F'): -2, ('S', 'G'): 0, ('S', 'H'): -1, ('S', 'I'): -2, ('S', 'K'): 0, ('S', 'L'): -2, ('S', 'N'): 1, ('S', 'P'): -1, ('S', 'Q'): 0, ('S', 'S'): 4, ('S', 'T'): 1, ('S', 'V'): -2, ('S', 'W'): -3, ('S', 'Y'): -2,
    ('T', 'A'): 0, ('T', 'D'): -1, ('T', 'E'): -1, ('T', 'F'): -2, ('T', 'G'): -2, ('T', 'H'): -2, ('T', 'I'): -1, ('T', 'K'): -1, ('T', 'L'): -1, ('T', 'N'): 0, ('T', 'P'): -1, ('T', 'Q'): -1, ('T', 'S'): 1, ('T', 'T'): 5, ('T', 'V'): 0, ('T', 'W'): -2, ('T', 'Y'): -2,
    ('V', 'A'): 0, ('V', 'D'): -3, ('V', 'E'): -2, ('V', 'F'): -1, ('V', 'G'): -3, ('V', 'H'): -3, ('V', 'I'): 3, ('V', 'K'): -2, ('V', 'L'): 1, ('V', 'N'): -3, ('V', 'P'): -2, ('V', 'Q'): -2, ('V', 'S'): -2, ('V', 'T'): 0, ('V', 'V'): 4, ('V', 'W'): -3, ('V', 'Y'): -1,
    ('W', 'A'): -3, ('W', 'D'): -4, ('W', 'E'): -3, ('W', 'F'): 1, ('W', 'G'): -2, ('W', 'H'): -2, ('W', 'I'): -3, ('W', 'K'): -3, ('W', 'L'): -2, ('W', 'N'): -4, ('W', 'P'): -4, ('W', 'Q'): -2, ('W', 'S'): -3, ('W', 'T'): -2, ('W', 'V'): -3, ('W', 'W'): 11, ('W', 'Y'): 2,
    ('Y', 'A'): -2, ('Y', 'D'): -3, ('Y', 'E'): -2, ('Y', 'F'): 3, ('Y', 'G'): -3, ('Y', 'H'): 2, ('Y', 'I'): -1, ('Y', 'K'): -2, ('Y', 'L'): -1, ('Y', 'N'): -2, ('Y', 'P'): -3, ('Y', 'Q'): -1, ('Y', 'S'): -2, ('Y', 'T'): -2, ('Y', 'V'): -1, ('Y', 'W'): 2, ('Y', 'Y'): 7,
}


def identity_score(a, b):
    return 1.0 if a == b else -1.0

def blosum62_score(a, b):
    return BLOSUM62_WIDE[a, b]

def pam250_score(a, b):
    return PAM250_SUB[a, b]

print("Varying the gap penalty (BLOSUM62 fixed):")
for gap_test in [-1.0, -4.0, -8.0, -12.0]:
    S_t, tr_t = global_align(seqA, seqB, gap_test, blosum62_score)
    oA, oB = traceback(seqA, seqB, S_t, tr_t)
    print(f"  gap={gap_test:>6.1f}  score={S_t[-1,-1]:>5.0f}   {oA} / {oB}")

print()
print("Varying the substitution matrix (gap=-4 fixed):")
for label, subst in [("identity", identity_score), ("BLOSUM62", blosum62_score), ("PAM250", pam250_score)]:
    S_t, tr_t = global_align(seqA, seqB, -4.0, subst)
    oA, oB = traceback(seqA, seqB, S_t, tr_t)
    print(f"  {label:>8s}  score={S_t[-1,-1]:>5.0f}   {oA} / {oB}")


### E-values: a database-search concept, shown against a real database

An **E-value** doesn't come from a two-sequence alignment score by
itself -- it's the expected number of chance hits *this good or better*
you'd see searching *a database of a given size*, which needs a real
database and BLAST's own statistics (Karlin-Altschul), not just the DP
recurrence above. So rather than inventing an E-value for the tiny
`seqA`/`seqB` pair, this builds a small **real** local BLAST database
from the same globin-family proteins used elsewhere in this course (HBB,
HBA1, MB, NGB, CYGB -- fetched from UniProt while building this
notebook) and runs a real `blastp` search against it, varying the same
two kinds of parameters as above (gap penalty, substitution matrix) to
see how the real, reported E-value actually responds.


In [ ]:
import shutil, subprocess, os

if shutil.which("blastp") is None:
    print("Installing NCBI BLAST+ (not preinstalled in Colab, unlike NumPy)...")
    subprocess.run(["apt-get", "install", "-y", "-qq", "ncbi-blast+"], check=True)
else:
    print("blastp already available, skipping install.")

# Real full-length sequences of the same globin-family proteins used
# elsewhere in this course (fetched from UniProt while building this
# notebook: HBB=P68871, HBA1=P69905, MB=P02144, NGB=Q9NPG2, CYGB=Q8WWM9).
GLOBIN_SEQS = {
    "HBB":  "MVHLTPEEKSAVTALWGKVNVDEVGGEALGRLLVVYPWTQRFFESFGDLSTPDAVMGNPKVKAHGKKVLGAFSDGLAHLDNLKGTFATLSELHCDKLHVDPENFRLLGNVLVCVLAHHFGKEFTPPVQAAYQKVVAGVANALAHKYH",
    "HBA1": "MVLSPADKTNVKAAWGKVGAHAGEYGAEALERMFLSFPTTKTYFPHFDLSHGSAQVKGHGKKVADALTNAVAHVDDMPNALSALSDLHAHKLRVDPVNFKLLSHCLLVTLAAHLPAEFTPAVHASLDKFLASVSTVLTSKYR",
    "MB":   "MGLSDGEWQLVLNVWGKVEADIPGHGQEVLIRLFKGHPETLEKFDKFKHLKSEDEMKASEDLKKHGATVLTALGGILKKKGHHEAEIKPLAQSHATKHKIPVKYLEFISECIIQVLQSKHPGDFGADAQGAMNKALELFRKDMASNYKELGFQG",
    "NGB":  "MERPEPELIRQSWRAVSRSPLEHGTVLFARLFALEPDLLPLFQYNCRQFSSPEDCLSSPEFLDHIRKVMLVIDAAVTNVEDLSSLEEYLASLGRKHRAVGVKLSSFSTVGESLLYMLEKCLGPAFTPATRAAWSQLYGAVVQAMSRGWDGE",
    "CYGB": "MEKVPGEMEIERRERSEELSEAERKAVQAMWARLYANCEDVGVAILVRFFVNFPSAKQYFSQFKHMEDPLEMERSPQLRKHACRVMGALNTVVENLHDPDKVSSVLALVGKAHALKHKVEPVYFKILSGVILEVVAEEFASDFPPETQRAWAKLRGLIYSHVTAAYKEVGWVQQVPNATTPPATLPSSGP",
}

os.makedirs("blast_db", exist_ok=True)
with open("blast_db/globins.fasta", "w") as f:
    for name, seq in GLOBIN_SEQS.items():
        f.write(f">{name}\n{seq}\n")

subprocess.run(
    ["makeblastdb", "-in", "blast_db/globins.fasta", "-dbtype", "prot", "-out", "blast_db/globindb"],
    check=True, capture_output=True,
)
with open("blast_db/query.fasta", "w") as f:
    f.write(f">query_HBB\n{GLOBIN_SEQS['HBB']}\n")

def run_blastp(extra_args):
    result = subprocess.run(
        ["blastp", "-query", "blast_db/query.fasta", "-db", "blast_db/globindb",
         "-outfmt", "6 sseqid evalue bitscore pident"] + extra_args,
        check=True, capture_output=True, text=True,
    )
    return result.stdout.strip().splitlines()

print("Real blastp hits, default (BLOSUM62, gapopen=11, gapextend=1):")
for line in run_blastp([]):
    print(" ", line)

print()
print("Varying the gap penalty (BLOSUM62 fixed) -- only certain (gapopen, gapextend)")
print("combinations are valid for a given matrix, per BLAST+'s own restrictions:")
for go, ge in [(11, 1), (9, 1), (7, 2)]:
    print(f"  gapopen={go}, gapextend={ge}:")
    for line in run_blastp(["-gapopen", str(go), "-gapextend", str(ge)]):
        print("   ", line)

print()
print("Varying the substitution matrix (default gap penalties):")
for matrix in ["BLOSUM62", "BLOSUM45", "PAM30"]:
    print(f"  matrix={matrix}:")
    for line in run_blastp(["-matrix", matrix]):
        print("   ", line)


Notice the real HBB-vs-HBB self-hit's E-value barely moves (it's an
exact match, overwhelmingly significant either way), but the weaker real
hits (HBA1, CYGB, NGB -- genuine but more distant relatives) shift
E-value by orders of magnitude as the gap penalty and substitution
matrix change -- exactly the same kind of parameter sensitivity Step 2's
first half showed on the tiny DP example, now visible in a real,
database-scale search. This is also a direct preview of Day 5, where
BLAST and E-values get their own full treatment.


## Step 3: a simple multiple sequence alignment

The book page's own "Multiple sequence alignment" section describes the
**progressive** strategy (align the closest pair first, then bring in
each further sequence guided by the pairwise similarities) but only
shows a guide tree, not an actual alignment. This builds one, on three
real, short windows (the same conserved `WGKV` motif region used
elsewhere on the Day 4 page) from three real proteins already used
throughout this course:

- **HBA1** (alpha-globin), **MB** (myoglobin), and **HBB** (beta-globin)
  -- residues 9-28 (HBA1/MB) or 10-29 (HBB) of each, fetched from
  UniProt while building this notebook.

This is a deliberately simplified, from-scratch demonstration of the
*progressive* strategy using the same pairwise `global_align` from Step
2 -- not a production MSA tool (a real tool like MUSCLE or Clustal Omega
handles profile-vs-profile scoring, affine gap costs, and iterative
refinement; this only aligns the closest pair, then merges in the third
sequence against whichever of the first two it's closer to).


In [ ]:
# The same conserved-motif windows used in the "Real substitution
# matrices" section of the teaching notebook, extended to a third
# sequence -- real 20-residue windows from UniProt, all containing the
# WGKV motif:
MSA_WINDOWS = {
    "HBA1": "TNVKAAWGKVGAHAGEYGAE",  # P69905, residues 9-28
    "MB":   "QLVLNVWGKVEADIPGHGQE",  # P02144, residues 9-28
    "HBB":  "SAVTALWGKVNVDEVGGEAL",  # P68871, residues 10-29
}

def pairwise_score(a, b):
    S_ab, _ = global_align(a, b, -6.0, blosum62_score)
    return S_ab[-1, -1]

# Step A: find the closest pair by real pairwise score.
names = list(MSA_WINDOWS)
best_pair, best_score = None, -np.inf
for i in range(len(names)):
    for j in range(i + 1, len(names)):
        s = pairwise_score(MSA_WINDOWS[names[i]], MSA_WINDOWS[names[j]])
        print(f"  pairwise score {names[i]}-{names[j]}: {s:.0f}")
        if s > best_score:
            best_score, best_pair = s, (names[i], names[j])
print(f"Closest pair: {best_pair[0]}-{best_pair[1]} (score {best_score:.0f}) -- align these first.")

n1, n2 = best_pair
n3 = [n for n in names if n not in best_pair][0]

S12, tr12 = global_align(MSA_WINDOWS[n1], MSA_WINDOWS[n2], -6.0, blosum62_score)
a1, a2 = traceback(MSA_WINDOWS[n1], MSA_WINDOWS[n2], S12, tr12)
profile = {n1: a1, n2: a2}

# Step B: which of n1/n2 is the third sequence more similar to? Align
# the third sequence against that one's RAW (ungapped) sequence.
ref = n1 if pairwise_score(MSA_WINDOWS[n3], MSA_WINDOWS[n1]) >= pairwise_score(MSA_WINDOWS[n3], MSA_WINDOWS[n2]) else n2
S3r, tr3r = global_align(MSA_WINDOWS[n3], MSA_WINDOWS[ref], -6.0, blosum62_score)
a3, a_ref_new = traceback(MSA_WINDOWS[n3], MSA_WINDOWS[ref], S3r, tr3r)
print(f"Aligning in {n3!r} against {ref!r} (its closer of the two already-aligned sequences).")

# Step C: merge -- walk the OLD aligned reference row and the FRESH
# aligned reference row together; wherever one has a gap the other
# doesn't, pad every other row accordingly. (A simplified version of the
# real "once a gap, always a gap" progressive-alignment merge rule.)
def merge_profile(profile_rows, ref_name, new_ref_aligned, new_seq_aligned):
    old_ref = profile_rows[ref_name]
    cols = {name: [] for name in profile_rows}
    cols["__NEW__"] = []
    i = j = 0
    while i < len(old_ref) or j < len(new_ref_aligned):
        old_c = old_ref[i] if i < len(old_ref) else None
        new_c = new_ref_aligned[j] if j < len(new_ref_aligned) else None
        if old_c == '-' and new_c != '-':
            for name, row in profile_rows.items():
                cols[name].append(row[i])
            cols["__NEW__"].append('-')
            i += 1
        elif new_c == '-' and old_c != '-':
            for name, row in profile_rows.items():
                cols[name].append('-')
            cols["__NEW__"].append(new_seq_aligned[j])
            j += 1
        else:
            for name, row in profile_rows.items():
                cols[name].append(row[i])
            cols["__NEW__"].append(new_seq_aligned[j])
            i += 1
            j += 1
    return {name: "".join(c) for name, c in cols.items()}

merged = merge_profile(profile, ref, a_ref_new, a3)
merged[n3] = merged.pop("__NEW__")

print()
print("Final progressive MSA:")
for name in names:
    print(f"  {name:5s} {merged[name]}")

msa_result = merged  # used by the visualization cell below


In [ ]:
import matplotlib.pyplot as plt

names_order = list(msa_result)
ncols = len(next(iter(msa_result.values())))

fig, ax = plt.subplots(figsize=(0.42 * ncols + 1.5, 0.6 * len(names_order) + 1))
for row, name in enumerate(names_order):
    seq = msa_result[name]
    for col, ch in enumerate(seq):
        others = [msa_result[n][col] for n in names_order if n != name]
        if ch == '-':
            color = "#f0f0f0"
        elif all(ch == o for o in others):
            color = "#8fd18f"
        else:
            color = "#ffe08a"
        ax.add_patch(plt.Rectangle((col, len(names_order) - 1 - row), 1, 1,
                                    facecolor=color, edgecolor="white"))
        ax.text(col + 0.5, len(names_order) - 1 - row + 0.5, ch,
                ha="center", va="center", fontsize=11, family="monospace")
ax.set_xlim(0, ncols)
ax.set_ylim(0, len(names_order))
ax.set_yticks([len(names_order) - 1 - i + 0.5 for i in range(len(names_order))])
ax.set_yticklabels(names_order)
ax.set_xticks([])
ax.set_title("Progressive MSA: real 20-residue windows around the conserved WGKV motif\n"
              "(green = fully conserved column, yellow = mismatch, grey = gap)")
plt.tight_layout()
plt.show()


## Done

Once Step 1's row checks and traceback check all say "correct," and
you've read through Steps 2 and 3's real output, you have everything the
Day 4 lab quiz on Canvas asks for.
